In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 3


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.7515603825449944
Epoch 2/100, Loss: 1.9104669392108917
Epoch 3/100, Loss: 1.8990277200937271
Epoch 4/100, Loss: 1.8212538361549377
Epoch 5/100, Loss: 1.7625631764531136
Epoch 6/100, Loss: 1.8159158825874329
Epoch 7/100, Loss: 1.8337312042713165
Epoch 8/100, Loss: 1.8432239815592766
Epoch 9/100, Loss: 1.801715448498726
Epoch 10/100, Loss: 1.9521240964531898
Epoch 11/100, Loss: 1.7448914423584938
Epoch 12/100, Loss: 1.820160612463951
Epoch 13/100, Loss: 1.9187781661748886
Epoch 14/100, Loss: 1.8736721351742744
Epoch 15/100, Loss: 1.7790972292423248
Epoch 16/100, Loss: 1.858100451529026


Epoch 17/100, Loss: 1.7972641810774803
Epoch 18/100, Loss: 1.8827858194708824
Epoch 19/100, Loss: 1.7391528561711311
Epoch 20/100, Loss: 1.8635438606142998
Epoch 21/100, Loss: 1.895273707807064
Epoch 22/100, Loss: 1.8618924841284752
Epoch 23/100, Loss: 1.7889448329806328
Epoch 24/100, Loss: 1.7419726848602295
Epoch 25/100, Loss: 1.7977091819047928
Epoch 26/100, Loss: 1.8700201362371445
Epoch 27/100, Loss: 1.8974376320838928
Epoch 28/100, Loss: 1.8559366762638092
Epoch 29/100, Loss: 1.8099554926156998
Epoch 30/100, Loss: 1.7221157625317574
Epoch 31/100, Loss: 1.848528191447258


Epoch 32/100, Loss: 1.804016150534153
Epoch 33/100, Loss: 1.8852509409189224
Epoch 34/100, Loss: 1.89764154702425
Epoch 35/100, Loss: 1.8015763312578201
Epoch 36/100, Loss: 1.7985130921006203
Epoch 37/100, Loss: 1.931572251021862
Epoch 38/100, Loss: 1.8058183118700981
Epoch 39/100, Loss: 1.8330544233322144
Epoch 40/100, Loss: 1.7564648762345314
Epoch 41/100, Loss: 1.826874129474163
Epoch 42/100, Loss: 1.7586960345506668
Epoch 43/100, Loss: 1.832995429635048
Epoch 44/100, Loss: 1.8580921068787575
Epoch 45/100, Loss: 1.7149827033281326
Epoch 46/100, Loss: 1.8503417745232582
Epoch 47/100, Loss: 1.7632745504379272


Epoch 48/100, Loss: 1.8211930692195892
Epoch 49/100, Loss: 1.7871555760502815
Epoch 50/100, Loss: 2.03624127805233
Epoch 51/100, Loss: 1.841316170990467
Epoch 52/100, Loss: 1.768935687839985
Epoch 53/100, Loss: 1.7524097487330437
Epoch 54/100, Loss: 1.7770256772637367
Epoch 55/100, Loss: 1.779124230146408
Epoch 56/100, Loss: 1.7822710573673248
Epoch 57/100, Loss: 1.9992993846535683
Epoch 58/100, Loss: 1.891787402331829
Epoch 59/100, Loss: 1.869819588959217
Epoch 60/100, Loss: 1.805646888911724
Epoch 61/100, Loss: 1.7944625206291676


Epoch 62/100, Loss: 1.8401687666773796
Epoch 63/100, Loss: 1.7542677819728851
Epoch 64/100, Loss: 1.8225600942969322
Epoch 65/100, Loss: 1.8033502995967865
Epoch 66/100, Loss: 1.8360947780311108
Epoch 67/100, Loss: 1.8618725910782814
Epoch 68/100, Loss: 1.8274338021874428
Epoch 69/100, Loss: 1.921421855688095
Epoch 70/100, Loss: 1.9003936871886253
Epoch 71/100, Loss: 1.698292389512062
Epoch 72/100, Loss: 1.8409631699323654
Epoch 73/100, Loss: 1.8799307346343994


...

Epoch 30/100, Loss: 2.8394449949264526
Epoch 31/100, Loss: 3.3680094331502914
Epoch 32/100, Loss: 3.093258172273636
Epoch 33/100, Loss: 3.3825193643569946
Epoch 34/100, Loss: 3.239716738462448
Epoch 35/100, Loss: 2.84143877774477
Epoch 36/100, Loss: 2.996789902448654
Epoch 37/100, Loss: 3.3833324909210205
Epoch 38/100, Loss: 3.2678209841251373
Epoch 39/100, Loss: 3.1179490983486176
Epoch 40/100, Loss: 2.756670691072941
Epoch 41/100, Loss: 3.3364897966384888
Epoch 42/100, Loss: 4.203021466732025
Epoch 43/100, Loss: 2.9476373344659805
Epoch 44/100, Loss: 3.1126283705234528
Epoch 45/100, Loss: 3.5232329815626144


Epoch 46/100, Loss: 2.876725323498249
Epoch 47/100, Loss: 3.2332330271601677
Epoch 48/100, Loss: 3.2340055406093597
Epoch 49/100, Loss: 3.249439150094986
Epoch 50/100, Loss: 2.7818220406770706
Epoch 51/100, Loss: 3.1941023617982864
Epoch 52/100, Loss: 3.2713362127542496
Epoch 53/100, Loss: 3.3078852593898773
Epoch 54/100, Loss: 3.2311978936195374
Epoch 55/100, Loss: 2.738066628575325
Epoch 56/100, Loss: 3.1258737295866013
Epoch 57/100, Loss: 3.127941206097603
Epoch 58/100, Loss: 3.113676443696022
Epoch 59/100, Loss: 2.812264032661915
Epoch 60/100, Loss: 3.9764060229063034
Epoch 61/100, Loss: 3.333815887570381
Epoch 62/100, Loss: 2.8706598430871964


Epoch 63/100, Loss: 3.166857086122036
Epoch 64/100, Loss: 3.2696578353643417
Epoch 65/100, Loss: 3.155875764787197
Epoch 66/100, Loss: 3.254523366689682
Epoch 67/100, Loss: 3.0339681804180145
Epoch 68/100, Loss: 2.9474261850118637
Epoch 69/100, Loss: 3.1353594809770584
Epoch 70/100, Loss: 2.7716362178325653
Epoch 71/100, Loss: 3.2452556639909744
Epoch 72/100, Loss: 3.254907451570034
Epoch 73/100, Loss: 3.5806362479925156
Epoch 74/100, Loss: 2.8939796090126038
Epoch 75/100, Loss: 3.0942001566290855
Epoch 76/100, Loss: 3.092727445065975
Epoch 77/100, Loss: 3.3802905455231667
Epoch 78/100, Loss: 3.2012377232313156


Epoch 79/100, Loss: 3.1878518760204315
Epoch 80/100, Loss: 4.023049421608448
Epoch 81/100, Loss: 2.7271576449275017
Epoch 82/100, Loss: 2.8689316883683205
Epoch 83/100, Loss: 2.9835325106978416
Epoch 84/100, Loss: 2.7281745076179504
Epoch 85/100, Loss: 3.739524982869625
Epoch 86/100, Loss: 3.203171283006668
Epoch 87/100, Loss: 2.842701569199562
Epoch 88/100, Loss: 2.953139975667
Epoch 89/100, Loss: 3.133902594447136
Epoch 90/100, Loss: 2.9068827480077744
Epoch 91/100, Loss: 3.0733401626348495
Epoch 92/100, Loss: 3.309524931013584
Epoch 93/100, Loss: 3.2334972620010376
Epoch 94/100, Loss: 3.003682255744934


Epoch 95/100, Loss: 3.167065054178238
Epoch 96/100, Loss: 3.0856754407286644
Epoch 97/100, Loss: 3.2245699018239975
Epoch 98/100, Loss: 2.77441792935133
Epoch 99/100, Loss: 3.011268526315689
Epoch 100/100, Loss: 3.5695887207984924
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5607
